# Aula 06 — Intervalos de Confiança

## Objetivos
1. Entender o conceito de **estimação por intervalo**.
2. Construir IC para a média (com $\sigma$ conhecido e desconhecido).
3. Construir IC para uma proporção.
4. Interpretar corretamente o "nível de confiança".

---

## 1. Estimação pontual vs. por intervalo

Uma **estimativa pontual** é um único número: $\bar{x} = 4{,}3$%.
Mas qual a chance dela ser exatamente igual à média populacional? Praticamente zero.

Um **intervalo de confiança** entrega uma faixa plausível:
> "Com 95% de confiança, a média populacional está entre 3,8% e 4,8%."

## 2. IC para a média — variância conhecida

Pelo TCL, $\bar{X} \sim N(\mu,\ \sigma^2/n)$. Então:

$$IC_{1-\alpha}(\mu) = \bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

Onde $z_{\alpha/2}$ é o quantil da Normal padrão. Para 95%: $z_{0{,}025} \approx 1{,}96$.

## 3. IC para a média — variância desconhecida (caso real)

Substituímos $\sigma$ por $s$ (desvio amostral) e usamos a **t de Student** com $n-1$ graus de liberdade:

$$IC_{1-\alpha}(\mu) = \bar{x} \pm t_{\alpha/2,\,n-1} \cdot \frac{s}{\sqrt{n}}$$

Quando $n$ é grande, $t \to z$.

## 4. IC para uma proporção

$$\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

(Aproximação válida quando $n\hat{p} \ge 5$ e $n(1-\hat{p}) \ge 5$.)

---

## ⚠️ Interpretação correta

**ERRADO:** "Há 95% de probabilidade do parâmetro estar no intervalo."

**CERTO:** "Se repetíssemos o procedimento muitas vezes, 95% dos intervalos
construídos conteriam o parâmetro verdadeiro."

A probabilidade está no **procedimento**, não no intervalo específico.

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

from utils.sidra_api import obter_ipca_mensal, obter_pib_per_capita_uf

## 5. IC para a média do IPCA dos últimos 24 meses

In [ ]:
df = obter_ipca_mensal(24)
x = df["variacao_mensal"].values
n = len(x)
media = x.mean()
s = x.std(ddof=1)  # desvio amostral (n-1)

# Nível de confiança 95%
alpha = 0.05
gl = n - 1
t_crit = stats.t.ppf(1 - alpha/2, gl)
erro = t_crit * s / np.sqrt(n)

li, ls = media - erro, media + erro
print(f"n = {n}")
print(f"Média amostral: {media:.4f}%")
print(f"Desvio-padrão amostral: {s:.4f}")
print(f"t crítico (gl={gl}, α/2=0.025): {t_crit:.4f}")
print(f"Erro-padrão da média: {s/np.sqrt(n):.4f}")
print(f"\nIC 95% para a média mensal do IPCA: [{li:.4f}%, {ls:.4f}%]")

### Usando `scipy` diretamente

In [ ]:
ic = stats.t.interval(0.95, df=gl, loc=media, scale=stats.sem(x))
print(f"IC 95% via scipy: [{ic[0]:.4f}%, {ic[1]:.4f}%]")

## 6. Comparando largura do IC para diferentes níveis de confiança

In [ ]:
niveis = [0.80, 0.90, 0.95, 0.99]
print(f"{'Nível':>6s} | {'LI':>8s} | {'LS':>8s} | {'Largura':>8s}")
print("-" * 40)
for nc in niveis:
    li_n, ls_n = stats.t.interval(nc, df=gl, loc=media, scale=stats.sem(x))
    print(f"{nc*100:5.0f}% | {li_n:8.4f} | {ls_n:8.4f} | {ls_n-li_n:8.4f}")
print("\nQuanto MAIOR a confiança, MAIS LARGO o intervalo. Trade-off clássico.")

## 7. IC para uma proporção

Pergunta: olhando os últimos 60 meses, qual a proporção de meses com inflação acima de 0,5%?

In [ ]:
df60 = obter_ipca_mensal(60)
sucessos = (df60["variacao_mensal"] > 0.5).sum()
n = len(df60)
p_hat = sucessos / n

z = stats.norm.ppf(0.975)
erro_p = z * np.sqrt(p_hat * (1 - p_hat) / n)
li_p, ls_p = p_hat - erro_p, p_hat + erro_p

print(f"Sucessos (IPCA > 0,5%): {sucessos}/{n}")
print(f"Proporção amostral: {p_hat:.3f}")
print(f"IC 95% para a proporção: [{li_p:.3f}, {ls_p:.3f}]")

## 8. Visualizando: simulação de 100 intervalos

Sortearemos 100 amostras aleatórias e construiremos 100 ICs. Cerca de 95 deveriam
conter a verdadeira média populacional.

In [ ]:
# População "verdadeira" (usaremos a Normal com parâmetros estimados do IPCA)
mu_pop = 0.45
sigma_pop = 0.40
n_amostra = 30

fig, ax = plt.subplots(figsize=(10, 8))
acertos = 0

for i in range(100):
    amostra = rng.normal(mu_pop, sigma_pop, n_amostra)
    m = amostra.mean()
    ic_inf, ic_sup = stats.t.interval(0.95, df=n_amostra-1,
                                       loc=m, scale=stats.sem(amostra))
    contem = ic_inf <= mu_pop <= ic_sup
    acertos += contem
    cor = "steelblue" if contem else "red"
    ax.plot([ic_inf, ic_sup], [i, i], color=cor, linewidth=1.2)
    ax.plot(m, i, "o", color=cor, markersize=3)

ax.axvline(mu_pop, color="black", linestyle="--", linewidth=2, label=f"μ verdadeiro = {mu_pop}")
ax.set_xlabel("Valor")
ax.set_ylabel("Réplica")
ax.set_title(f"100 ICs 95% — {acertos} contêm a verdadeira média (esperado: ~95)",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula06_simulacao_ic.png"), dpi=150, bbox_inches="tight")
plt.show()

## Exercícios

1. Construa IC 90% e 99% para a média do PIB per capita das UFs.
2. Calcule a proporção de UFs com PIB per capita acima da média nacional e construa IC 95%.
3. O tamanho de amostra para um IC com erro máximo de 0,1% (na média do IPCA) seria quanto?
   *(Dica: isole $n$ na fórmula do erro.)*

**Próxima aula:** Testes de hipótese (z e t).